# PaliGemma 2 — örnek testler

**Colab hazırlığı:** T4 GPU · Secrets'ta `HF_TOKEN` · [paligemma2-3b-mix-448](https://huggingface.co/google/paligemma2-3b-mix-448) lisansı kabul · `data/` klasöründeki `fis1-3.jpeg` ve `fis1-3.txt` dosyaları `/content`'e yüklü.

Şekiller `sekiller/` klasörüne kaydedilir, en son hücre hepsini indirir.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import os, re, json, gc, requests, torch
import numpy as np
from io import BytesIO
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from matplotlib import font_manager
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
device = "cuda"
os.makedirs("sekiller", exist_ok=True)
sonuc = {}  # yazıya girecek sayılar, en sonda sekiller/sonuclar.json

def load(url):
    r = requests.get(url, headers={"User-Agent": "paligemma-colab-demo"}, timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

imgs = {
    "kediler": load("http://images.cocodataset.org/val2017/000000039769.jpg"),
    "salon":   load("http://images.cocodataset.org/val2017/000000000139.jpg"),
    "araba":   load("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"),
    "ari":     load("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"),
}

In [ ]:
# Şekil stili ve yardımcılar
MAVI, TURUNCU, MUREKKEP, GRI = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRI,
    "xtick.color": GRI, "ytick.color": GRI, "axes.labelcolor": GRI, "text.color": MUREKKEP,
    "grid.color": "#e6e5e0", "axes.axisbelow": True, "legend.frameon": False,
    "figure.facecolor": "white", "savefig.dpi": 200, "savefig.bbox": "tight"})

from matplotlib.ticker import FuncFormatter
tr = lambda v, d=2: f"{v:.{d}f}".replace(".", ",")
TR_EKSEN = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))

def kaydet(fig, ad):
    fig.savefig(f"sekiller/{ad}.png")
    plt.show()

def sekil_siglip_clip(metinler, sig, clip):
    y, h = np.arange(len(metinler))[::-1], 0.38
    fig, ax = plt.subplots(figsize=(8.5, 0.55 * len(metinler) + 1.4))
    for dy, v, renk, ad in [(h / 2, sig, MAVI, "SigLIP, sigmoid"), (-h / 2, clip, TURUNCU, "CLIP, softmax")]:
        ax.barh(y + dy, v, h * 0.92, color=renk, label=f"{ad} (toplam {tr(sum(v))})")
        for yy, vv in zip(y + dy, v):
            if vv >= 0.01:
                ax.text(vv + 0.01, yy, tr(vv), va="center", fontsize=9, color=GRI)
    ax.set_yticks(y, metinler)
    ax.set_xlim(0, 1.1)
    ax.xaxis.grid(True)
    ax.xaxis.set_major_formatter(TR_EKSEN)
    ax.set_xlabel("olasılık")
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
    ax.set_title("Aynı fotoğraf, farklı açıklamalar", pad=30)
    return fig

def sekil_esik(metinler, cos, esik):
    y = np.arange(len(metinler))[::-1]
    fig, ax = plt.subplots(figsize=(8.5, 0.45 * len(metinler) + 1.4))
    ax.axvline(esik, color=TURUNCU, lw=2, ls="--", zorder=2)
    ax.scatter(cos, y, s=70, zorder=3, color=[MAVI if c >= esik else GRI for c in cos])
    ax.text(esik, y.max() + 0.7, f"  %50 eşiği: −b/t = {tr(esik, 3)}", fontsize=10, va="bottom")
    ax.set_yticks(y, metinler)
    ax.set_ylim(-0.7, y.max() + 1.4)
    ax.xaxis.grid(True)
    ax.xaxis.set_major_formatter(TR_EKSEN)
    ax.set_xlabel("resim ile metin arasındaki kosinüs benzerliği")
    ax.set_title("SigLIP \"eşleşiyor\" demek için ne kadar benzerlik istiyor?")
    return fig

def lev(a, b):
    onceki = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        simdi = [i]
        for j, cb in enumerate(b, 1):
            simdi.append(min(onceki[j] + 1, simdi[j - 1] + 1, onceki[j - 1] + (ca != cb)))
        onceki = simdi
    return onceki[-1]

def cer(tahmin, dogru):
    """Karakter hata oranı. Satır sonları ve boşluklar tek boşluğa indirilir, sıra önemlidir."""
    t, d = " ".join(tahmin.split()), " ".join(dogru.split())
    return lev(t, d) / len(d)

def kelime_isabeti(tahmin, dogru):
    """Doğru metindeki kelimelerin kaçı çıktıda birebir geçiyor. Sıra önemsiz."""
    d, t = Counter(dogru.split()), Counter(tahmin.split())
    return sum((d & t).values()) / sum(d.values())

assert lev("kitten", "sitting") == 3 and cer("a  b\nc", "a b c") == 0
assert kelime_isabeti("b a", "a b") == 1 and kelime_isabeti("a", "a b") == 0.5

def sekil_ocr(olcu):
    adlar = list(olcu["224"])
    x, w = np.arange(len(adlar)), 0.38
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, i, baslik in [(axes[0], 0, "Karakter hata oranı (düşük daha iyi)"),
                          (axes[1], 1, "Doğru okunan kelime oranı (yüksek daha iyi)")]:
        for dx, r, renk in [(-w / 2, "224", TURUNCU), (w / 2, "448", MAVI)]:
            v = [olcu[r][k][i] for k in adlar]
            ax.bar(x + dx, v, w * 0.92, color=renk, label=f"{r} piksel")
            for xx, vv in zip(x + dx, v):
                ax.text(xx, vv, tr(vv), ha="center", va="bottom", fontsize=8.5, color=GRI)
        ax.set_xticks(x, [a.replace(" ", "\n") for a in adlar])
        ax.yaxis.grid(True)
        ax.yaxis.set_major_formatter(TR_EKSEN)
        ax.set_title(baslik, fontsize=12)
    axes[1].set_ylim(0, 1.08)
    axes[0].legend(loc="upper left")
    return fig

def sekil_gorus(tam, yakin):
    """tam, yakin: [(başlık, HxWx3 dizi), (başlık, dizi)]. Üst satır tam görüntü, alt satır yakın plan."""
    oran = yakin[0][1].shape[0] / yakin[0][1].shape[1]
    fig, axes = plt.subplots(2, 2, figsize=(10, 5 * (1 + oran) + 0.8),
                             gridspec_kw={"height_ratios": [1, oran]})
    for ax, (baslik, a) in zip(axes.flat, tam + yakin):
        ax.imshow(a, interpolation="nearest")
        ax.set_title(baslik, fontsize=11)
        ax.axis("off")
    return fig

## 1. SigLIP ve CLIP

In [ ]:
from transformers import AutoModel, AutoProcessor, CLIPModel, CLIPProcessor

sig = AutoModel.from_pretrained("google/siglip-so400m-patch14-384").to(device).eval()
sig_p = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_p = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def skorla(metinler, image):
    x = sig_p(text=metinler, images=image, padding="max_length", return_tensors="pt").to(device)  # SigLIP böyle eğitildi
    sig_logit = sig(**x).logits_per_image[0].float().cpu()
    x = clip_p(text=metinler, images=image, padding=True, return_tensors="pt").to(device)
    clip_prob = clip(**x).logits_per_image.softmax(-1)[0].float().cpu()
    return sig_logit, torch.sigmoid(sig_logit), clip_prob

def tablo(metinler, s, c):
    print(f"{'metin':52s} {'SigLIP':>7s} {'CLIP':>7s}")
    for t, a, b in zip(metinler, s, c):
        print(f"{t:52s} {a:7.3f} {b:7.3f}")
    print(f"{'TOPLAM':52s} {s.sum():7.3f} {c.sum():7.3f}")

METIN5 = ["a photo of a cat", "a photo of two cats", "a photo of an animal", "a photo of a dog", "a photo of a car"]
_, sig5, clip5 = skorla(METIN5, imgs["kediler"])
tablo(METIN5, sig5, clip5)

Üç doğru açıklama daha ekliyoruz. SigLIP'te ilk beşin değeri değişmemeli, CLIP'te değişmeli.

In [ ]:
METIN8 = METIN5 + ["a photo of cats", "two cats sleeping on a pink blanket",
                   "two cats lying on a couch next to remote controls"]
sig_logit8, sig8, clip8 = skorla(METIN8, imgs["kediler"])
tablo(METIN8, sig8, clip8)
print("\nİlk beşin değeri değişti mi?  SigLIP:", not torch.allclose(sig8[:5], sig5, atol=1e-4),
      "  CLIP:", not torch.allclose(clip8[:5], clip5, atol=1e-4))

t, b = sig.logit_scale.exp().item(), sig.logit_bias.item()
cos = ((sig_logit8 - b) / t).tolist()
esik = -b / t  # sigmoid'in 0,5'i geçtiği kosinüs
print(f"\nÖğrenilen sıcaklık t = {t:.1f}, bias b = {b:.2f}  →  %50 için kosinüs ≥ {esik:.3f}")
for m, c, l in zip(METIN8, cos, sig_logit8):
    print(f"{m:52s} kosinüs {c:6.3f}   logit {l:6.2f}")

sonuc["siglip"] = {"t": t, "b": b, "esik": esik, "metinler": METIN8, "kosinus": cos,
                   "sigmoid": sig8.tolist(), "clip8": clip8.tolist(), "clip5": clip5.tolist()}

In [ ]:
kaydet(sekil_siglip_clip(METIN8, sig8.tolist(), clip8.tolist()), "siglip_clip")
kaydet(sekil_esik(METIN8, cos, esik), "siglip_esik")

In [ ]:
del sig, clip
gc.collect(); torch.cuda.empty_cache()

## PaliGemma 2 (3B, 448 piksel)

In [ ]:
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma2-3b-mix-448", torch_dtype=torch.bfloat16, device_map="auto").eval()
proc448 = proc = PaliGemmaProcessor.from_pretrained("google/paligemma2-3b-mix-448")
proc224 = PaliGemmaProcessor.from_pretrained("google/paligemma2-3b-mix-224")  # şimdilik yalnız işlemci

def ask(img, prompt, max_new_tokens=100, raw=False):
    """raw=True: <loc>/<seg> token'ları silinmez."""
    x = proc(text="<image>" + prompt, images=img, return_tensors="pt").to(model.device)
    x["pixel_values"] = x["pixel_values"].to(torch.bfloat16)
    n = x["input_ids"].shape[1]
    with torch.inference_mode():
        out = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.decode(out[0][n:], skip_special_tokens=not raw).replace("<eos>", "").replace("<pad>", "").strip()

print(ask(imgs["araba"], "caption en"))

## 2. Görsel token sayısı ve prefix-LM maskesi

In [ ]:
img_tok = proc.tokenizer.convert_tokens_to_ids("<image>")
for ad, p in [("224", proc224), ("448", proc448)]:
    ids = p(text="<image>caption en", images=imgs["araba"], return_tensors="pt")["input_ids"][0]
    print(f"{ad} piksel → görsel token: {(ids == img_tok).sum().item()}, toplam: {len(ids)}")

In [ ]:
n_img, n_txt, n_ans = 4, 3, 4
n_pre, N = n_img + n_txt, n_img + n_txt + n_ans
mask = torch.tril(torch.ones(N, N))
mask[:n_pre, :n_pre] = 1  # prefix kendi içinde tam dikkat

etiket = [f"img{i}" for i in range(n_img)] + [f"p{i}" for i in range(n_txt)] + [f"a{i}" for i in range(n_ans)]
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(mask, cmap="Blues")
ax.set_xticks(range(N), etiket, rotation=90); ax.set_yticks(range(N), etiket)
ax.axhline(n_pre - .5, color=TURUNCU); ax.axvline(n_pre - .5, color=TURUNCU)
ax.set_xlabel("bakılan (key)"); ax.set_ylabel("bakan (query)")
ax.set_title("Prefix-LM maskesi (mavi = görebilir)", fontsize=12)
kaydet(fig, "prefix_lm_maskesi")

## 3. Altyazı: İngilizce ve Türkçe

In [ ]:
for ad in imgs:
    print(f"{ad:8s} EN: {ask(imgs[ad], 'caption en')}\n{'':8s} TR: {ask(imgs[ad], 'caption tr')}")

## 4. Görsel soru-cevap

In [ ]:
sorular = [
    ("kediler", "answer en how many cats are there?"),
    ("kediler", "answer en what color is the couch?"),
    ("kediler", "answer en what is next to the cats?"),
    ("salon",   "answer en is there a television in the room?"),
    ("salon",   "answer en how many people are in the room?"),
    ("salon",   "answer tr odada kaç sandalye var?"),
    ("araba",   "answer en what color is the car?"),
]
for ad, q in sorular:
    print(f"[{ad}] {q}  →  {ask(imgs[ad], q, max_new_tokens=20)}")

## 5. Nesne tespiti: `<loc>` token'ları

In [ ]:
def parse_detections(text, w, h):
    """'<locYYYY>x4 etiket ; ...' → [(etiket, (x0, y0, x1, y1)), ...]"""
    out = []
    for m in re.finditer(r"((?:<loc\d{4}>){4})\s*([^;<]+)", text):
        y0, x0, y1, x1 = [int(v) / 1024 for v in re.findall(r"\d{4}", m.group(1))]
        out.append((m.group(2).strip(), (x0 * w, y0 * h, x1 * w, y1 * h)))
    return out

_t = "<loc0000><loc0000><loc0512><loc0256> cat ; <loc0100><loc0200><loc0300><loc0400> remote"
assert parse_detections(_t, 1024, 1024) == [("cat", (0, 0, 256, 512)), ("remote", (200, 100, 400, 300))]

def draw(img, dets):
    im = img.copy(); d = ImageDraw.Draw(im)
    for label, box in dets:
        d.rectangle(box, outline="red", width=4)
        d.text((box[0] + 4, box[1] + 4), label, fill="red")
    return im

isler = [("kediler", "detect cat ; remote control"), ("salon", "detect person ; chair ; tv"), ("ari", "detect bee ; flower")]
fig, axes = plt.subplots(1, len(isler), figsize=(18, 6))
for ax, (ad, prompt) in zip(axes, isler):
    ham = ask(imgs[ad], prompt, raw=True)
    print(f"[{ad}] {ham}")
    ax.imshow(draw(imgs[ad], parse_detections(ham, *imgs[ad].size))); ax.set_title(prompt, fontsize=11); ax.axis("off")
kaydet(fig, "tespit")

## 6. OCR: 224 ve 448 piksel

Doğru metinler `fis*.txt` dosyalarında. İki ölçü: **karakter hata oranı** (sıra önemli, düşük daha iyi) ve **doğru okunan kelime oranı** (sıra önemsiz, yüksek daha iyi). Green Salads fişi dosyada yan duruyor; yan ve düz hâli ayrı ayrı okutuluyor.

In [ ]:
font = ImageFont.truetype(font_manager.findfont("DejaVu Sans"), 52)
def tabela(satirlar, size=(896, 448)):
    im = Image.new("RGB", size, "white"); d = ImageDraw.Draw(im)
    for i, s in enumerate(satirlar):
        d.text((40, 50 + i * 90), s, fill="black", font=font)
    return im

TABELA = ["Dikkat! Çığ tehlikesi", "Şişli – Üsküdar 12 km", "Öğrenci İndirimi %20"]
oku = lambda f: open(f, encoding="utf-8").read()

ocr_imgs = {"Starbucks": Image.open("fis1.jpeg").convert("RGB"),
            "Anadolu": Image.open("fis3.jpeg").convert("RGB"),
            "Green Salads (yan)": Image.open("fis2.jpeg").convert("RGB")}
ocr_imgs["Green Salads (düz)"] = ocr_imgs["Green Salads (yan)"].rotate(90, expand=True)
ocr_imgs["Tabela"] = tabela(TABELA)

GT = {"Starbucks": oku("fis1.txt"), "Anadolu": oku("fis3.txt"),
      "Green Salads (yan)": oku("fis2.txt"), "Green Salads (düz)": oku("fis2.txt"), "Tabela": "\n".join(TABELA)}

fig, axes = plt.subplots(1, len(ocr_imgs), figsize=(18, 4.5))
for ax, (ad, im) in zip(axes, ocr_imgs.items()):
    ax.imshow(im); ax.set_title(ad, fontsize=10); ax.axis("off")
plt.show()

In [ ]:
MAX_TOKENS = 512  # fişler uzun; Gemma rakamları tek tek token'lar
ocr = {"448": {ad: ask(im, "ocr", max_new_tokens=MAX_TOKENS) for ad, im in ocr_imgs.items()}}

# 224 modeline geç (ikisi birden T4'e sığmaz)
del model; gc.collect(); torch.cuda.empty_cache()
model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma2-3b-mix-224", torch_dtype=torch.bfloat16, device_map="auto").eval()
proc = proc224
ocr["224"] = {ad: ask(im, "ocr", max_new_tokens=MAX_TOKENS) for ad, im in ocr_imgs.items()}

In [ ]:
olcu = {r: {ad: (cer(ocr[r][ad], GT[ad]), kelime_isabeti(ocr[r][ad], GT[ad])) for ad in GT} for r in ["224", "448"]}
print(f"{'':20s} {'hata 224':>9s} {'hata 448':>9s} {'kelime 224':>11s} {'kelime 448':>11s}")
for ad in GT:
    print(f"{ad:20s} {olcu['224'][ad][0]:9.2f} {olcu['448'][ad][0]:9.2f} {olcu['224'][ad][1]:11.2f} {olcu['448'][ad][1]:11.2f}")

sonuc["ocr"], sonuc["ocr_olcu"] = ocr, olcu
kaydet(sekil_ocr(olcu), "ocr_224_448")

In [ ]:
# Modelin gördüğü: işlemcinin kareye sıkıştırdığı resim
def goruntu(p, im):
    px = p.image_processor(im, return_tensors="pt")["pixel_values"][0]
    return ((px.permute(1, 2, 0).float() + 1) / 2).clamp(0, 1).numpy()

def yakin(a, y0=0.80, y1=0.97, x0=0.05, x1=0.80):  # fişin alt kısmındaki küçük yazılar
    h, w = a.shape[:2]
    return a[int(y0 * h):int(y1 * h), int(x0 * w):int(x1 * w)]

im = ocr_imgs["Starbucks"]
g224, g448 = goruntu(proc224, im), goruntu(proc448, im)
kaydet(sekil_gorus([("Model 224×224 görüyor", g224), ("Model 448×448 görüyor", g448)],
                   [("224, alt kısım yakından", yakin(g224)), ("448, alt kısım yakından", yakin(g448))]),
       "fis_modelin_gordugu")

Ham çıktılar

In [ ]:
for ad in GT:
    print(f"━━ {ad}\n── 224:\n{ocr['224'][ad]}\n── 448:\n{ocr['448'][ad]}\n")

In [ ]:
json.dump(sonuc, open("sekiller/sonuclar.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
!zip -qr sekiller.zip sekiller
from google.colab import files
files.download("sekiller.zip")